In [9]:
import mario
import yaml
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
years = range(2023,2024)   
versions = [
    'v1.0', 
    # 'v2.0'
    ]  # versions to parse

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

In [10]:
footprints = pd.DataFrame()
ghgs = [
    "Carbon dioxide, fossil (air - Emiss)",
    "CH4 (air - Emiss)",
    "N2O (air - Emiss)",
]

for version in versions:
    for year in years:
        print(f"Parsing version {version} for year {year}...", end="\n")
        db = mario.parse_from_txt(
            path = os.path.join(paths['export'], version, str(year),"flows"),
            mode = "flows",
            table = 'SUT',
        )

        print("DONE! Calculating footprints...", end="\n")
        f = db.f.loc[ghgs,:]
        f = f.T
        f['GHG'] = f['Carbon dioxide, fossil (air - Emiss)'] + \
                   f['CH4 (air - Emiss)']*25 + \
                   f['N2O (air - Emiss)']*298

        f.columns.names = ['Substances']
        f = f.stack().to_frame()
        f.columns = ['Value']
        f.reset_index(inplace=True)
        f['Year'] = year
        f['Version'] = version

        footprints = pd.concat([footprints, f], axis=0, ignore_index=True)
        print("DONE!\n")

Parsing version v1.0 for year 2023...


Database: to calculate f following matrices are need.
['e'].Trying to calculate dependencies.


DONE! Calculating footprints...


Database: to calculate f following matrices are need.
['w'].Trying to calculate dependencies.
Database: to calculate w following matrices are need.
['z'].Trying to calculate dependencies.


DONE!



Add footprints from EXIOBASE 3.3.18 raw

In [ ]:
version = "EXIOBASE 3.3.18"
year = 2011


print(f"Parsing version {version} for year {year}...", end="\n")
db = mario.parse_from_txt(
    paths['raw'], 
    table='SUT', 
    mode='flows'
)


print("DONE! Aggregating EE...", end="\n")
db.aggregate("support/aggregate_ee.xlsx",ignore_nan=True)


print("DONE! Calculating footprints...", end="\n")
f = db.f.loc[ghgs,:]
f = f.T
f['GHG'] = f['Carbon dioxide, fossil (air - Emiss)'] + \
            f['CH4 (air - Emiss)']*25 + \
            f['N2O (air - Emiss)']*298


f.columns.names = ['Substances']
f = f.stack().to_frame()
f.columns = ['Value']
f.reset_index(inplace=True)
f['Year'] = year
f['Version'] = version


footprints = pd.concat([footprints, f], axis=0, ignore_index=True)
print("DONE!\n")

In [3]:
footprints.to_csv(
    paths['export']+"/_results/Footprints.csv"
)

In [12]:
footprints_old = pd.read_csv(paths['export']+"/_results/Footprints.csv")

In [13]:
footprints_old.query("Region == 'IT' & Level=='Commodity' & Item=='Electricity' & Substances == 'GHG'")

,Unnamed: 0.1,Unnamed: 0,Region,Level,Item,Substances,Value,Year,Version
48467,48467,48467.0,IT,Commodity,Electricity,GHG,114.768939,2024,v1.0
115667,115667,115667.0,IT,Commodity,Electricity,GHG,116.689645,2025,v1.0
183151,183151,183151.0,IT,Commodity,Electricity,GHG,114.047510,2024,v2.0
250735,250735,250735.0,IT,Commodity,Electricity,GHG,115.946549,2025,v2.0
318035,318035,318035.0,IT,Commodity,Electricity,GHG,169.810291,2011,EXIOBASE 3.3.18
385519,385519,385519.0,IT,Commodity,Electricity,GHG,127.853284,2023,v2.0
452819,452819,NaN,IT,Commodity,Electricity,GHG,128.665075,2023,v1.0
